# Lekcija 22: Stereo podudaranje

Dvije kamere koje promatraju isti prizor iz malo različitih položaja vide iste 3D točke pomaknute za različite iznose ovisno o dubini &mdash; bliže točke pomiču se više, udaljenije manje. **Stereo podudaranje** pronalazi te pomake (**disparitet**) na svakom pikselu, koji se triangulacijom pretvaraju u dubinu. Ključan je problem korespondencija: za svaki piksel na lijevoj slici pronaći odgovarajući piksel na desnoj. Ova lekcija izgrađuje **postupak podudaranja blokova** koji to rješava uspoređivanjem malih područja pomoću zbroja kvadrata razlika (SSD) &mdash; blizak srodnik optičkog toka iz lekcije 21, ali ograničen na 1D pretraživanje jednog retka umjesto cijelog 2D susjedstva.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# shared colormap: invalid/masked-out disparities (NaN) render as gray instead of viridis's default
invalid_cmap = plt.cm.viridis.copy()
invalid_cmap.set_bad('gray')

## Rektificirani stereo: zašto je pretraživanje 1D

Za dvije kamere postavljene jednu uz drugu, usmjerene jednako i s paralelnim ravninama slika (**rektificirani** stereo par &mdash; stvarni se postavi kamera kalibriraju i slike preslikavaju kako bi to približno ostvarili), vrijedi temeljna činjenica epipolarne geometrije: odgovarajuća točka bilo kojeg piksela lijeve slike leži **u istom retku** desne slike. Time se traženje podudaranja svodi s 2D pretraživanja slike na 1D pretraživanje jednog retka, a vodoravni pomak između dvaju odgovarajućih položaja je **disparitet** $d$.

Disparitet je povezan s dubinom izrazom

$$Z = \frac{f \cdot B}{d}$$

gdje je $f$ žarišna duljina, a $B$ osnovica (udaljenost između središta dviju kamera). Za rektificirani par dubina je **obrnuto proporcionalna disparitetu**: bliski objekti imaju veliki disparitet, udaljeni mali, a beskonačno udaljeni objekt ima disparitet nula.

## Sintetički stereo par s poznatim stvarnim vrijednostima

Izgradimo lijevu sliku isključivo od nasumične teksture (kako bi svako područje bilo lokalno prepoznatljivo &mdash; bez dvosmislenosti problema otvora), a desnu konstruiramo pomicanjem svakog piksela ulijevo za njegov stvarni disparitet. Postavljamo tri različite stalne vrijednosti za tri dubinske „ravnine”: pozadinu i dva bliža pravokutnika. Bliže površine pomiču se više od udaljenih pa s jedne strane otkrivaju piksele pozadine koji uopće nemaju odgovarajući piksel na lijevoj slici &mdash; **okluziju**, odnosno zaklanjanje. Popunjavamo je novim nasumičnim šumom kako bi svaki piksel desne slike ipak imao neku vrijednost.

In [ ]:
rng = np.random.default_rng(0)
h, w = 150, 200
left = rng.integers(0, 255, (h, w)).astype(np.uint8)

true_disparity = np.full((h, w), 5, dtype=np.int32)     # background         (disparity =  5)
true_disparity[30:120, 50:150] = 15                     # a close rectangle  (disparity = 15)
true_disparity[60:90, 80:120] = 25                      # a closer rectangle (disparity = 25)

right = np.full((h, w), -1, dtype=np.int32)
for y in range(h):
    for x in range(w):
        xr = x - true_disparity[y, x]
        if 0 <= xr < w:
            right[y, xr] = left[y, x]
occluded = right == -1   # positions no left pixel maps to: disocclusions, filled with fresh noise
right[occluded] = rng.integers(0, 255, occluded.sum())
right = right.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(left, cmap='gray')
axes[0].set_title('Left image', fontsize=9)
axes[1].imshow(right, cmap='gray')
axes[1].set_title('Right image', fontsize=9)
im = axes[2].imshow(true_disparity, cmap='viridis')
axes[2].set_title('Ground-truth disparity', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

## Podudaranje blokova duž retka

Za svaki piksel lijeve slike pomičemo mali prozor duž *istog retka* desne slike preko raspona kandidata za disparitet i zadržavamo disparitet s najmanjim zbrojem kvadrata razlika. Implementacija u nastavku ima tri ugniježđene Python petlje, pa je spora &mdash; pokrećemo je na malom izrezu umjesto na cijeloj slici.

In [ ]:
def block_match_stereo(left, right, block_size=7, max_disp=30):
    h, w = left.shape
    half = block_size // 2
    left_f, right_f = left.astype(np.float64), right.astype(np.float64)
    disparity_map = np.zeros((h, w), dtype=np.float64)

    for y in range(half, h - half):
        for x in range(half, w - half):
            left_patch = left_f[y - half:y + half + 1, x - half:x + half + 1]
            best_d, best_cost = 0, np.inf
            for d in range(max_disp + 1):
                xr = x - d
                if xr - half < 0:
                    break
                right_patch = right_f[y - half:y + half + 1, xr - half:xr + half + 1]
                cost = ((left_patch - right_patch)**2).sum()
                if cost < best_cost:
                    best_cost, best_d = cost, d
            disparity_map[y, x] = best_d
    return disparity_map

# a small crop, since the pure-Python pixel loop is slow
crop = np.s_[0:100, 0:100]
manual_disp = block_match_stereo(left[crop], right[crop], block_size=7, max_disp=30)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(left[crop], cmap='gray')
axes[0].set_title(f'Left crop ({left[crop].shape[0]}x{left[crop].shape[1]})', fontsize=9)
axes[1].imshow(true_disparity[crop], cmap='viridis', vmin=0, vmax=30)
axes[1].set_title(f'True disparity ({true_disparity[crop].shape[0]}x{true_disparity[crop].shape[1]})', fontsize=9)
im = axes[2].imshow(manual_disp, cmap='viridis', vmin=0, vmax=30)
axes[2].set_title(f'Recovered disparity ({manual_disp.shape[0]}x{manual_disp.shape[1]})', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

Rub duž granica izreza može se zanemariti: ondje klizni prozor nema dovoljno prostora za pretraživanje, rubni učinak sličan obradi granica kod konvolucije (lekcija 10). Daleko od tog ruba i unutar pravokutnika, određeni disparitet gotovo savršeno odgovara stvarnim vrijednostima, jasno razdvajajući sve tri dubinske ravnine. Iznimka je pojas točkastog šuma uz rub svakog pravokutnika: prozor koji prelazi dubinsku granicu miješa piksele dvaju različitih stvarnih dispariteta, pa nijedan pojedini kandidat ne odgovara cijelom prozoru i podudaranje postaje nepouzdano &mdash; najava problema okluzije koji obrađujemo sljedeći. Stvarne fotografije još su zahtjevnije od ovog demonstracijskog slučaja: ponavljajuće teksture, ravnomjerna područja, zrcalni odsjaji i sjenčanje dodaju stvarnu dvosmislenost uz okluziju, pa je zato stereo podudaranje u praksi teško.

## Otkrivanje pogrešnih podudaranja: provjera dosljednosti lijevo–desno

Okluzije i dvosmislenost dubinskih granica stvaraju pogrešna podudaranja, poput gornjeg točkastog šuma. Rješenje: podudarati i slijeva nadesno i zdesna nalijevo, a zatim zadržati samo piksele za koje se dva rezultata slažu, odbacujući ostale kao nepouzdane. Računanje dispariteta zdesna nalijevo ne zahtijeva novi kod &mdash; vodoravno zrcalite obje slike, zamijenite koja igra ulogu „lijeve”, pokrenite isti postupak podudaranja, a zatim ponovno zrcalite rezultat. Piksel ne prolazi provjeru kada se dva smjera razlikuju za više od približno jednog piksela. To je osobito učinkovito za otkrivanje okluzija: piksel skriven iza bliže površine u drugom pogledu ondje uopće nema stvarno podudaranje.

In [ ]:
block_size, max_disp = 7, 30
crop_lr = np.s_[0:100, 0:100]
left_c, right_c = left[crop_lr], right[crop_lr]

disp_lr = block_match_stereo(left_c, right_c, block_size=block_size, max_disp=max_disp)
disp_rl = block_match_stereo(right_c[:, ::-1], left_c[:, ::-1], block_size=block_size, max_disp=max_disp)[:, ::-1]

h_c, w_c = disp_lr.shape
consistent = np.zeros((h_c, w_c), dtype=bool)
for y in range(h_c):
    for x in range(w_c):
        d = int(disp_lr[y, x])
        xr = x - d
        if 0 <= xr < w_c:
            consistent[y, x] = abs(disp_rl[y, xr] - d) <= 1

# keep the full crop (no trimming) so the border-truncation effect is visible directly,
# alongside the occlusion-driven inconsistency, in the figure below
disp_checked = np.where(consistent, disp_lr, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, d, title in zip(axes, [disp_lr, disp_rl, disp_checked],
                         ['Left-to-right disparity', 'Right-to-left disparity', 'After consistency check']):
    im = ax.imshow(d, cmap=invalid_cmap, vmin=0, vmax=max_disp)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

print(f'pixels marked inconsistent: {(~consistent).sum()} / {consistent.size}')

Mnogi se pikseli slažu u oba smjera i prolaze provjeru. Oni koji je ne prolaze obično su uzrokovani okluzijom (velika siva područja uz okomite rubove pravokutnika) ili konačnom veličinom prozora (mala siva područja blizu vodoravnih rubova pravokutnika).

## Isti postupak u punoj rezoluciji s OpenCV-jem

`cv2.StereoBM` implementira istu ideju podudaranja blokova (uz određena poboljšanja učinkovitosti i naknadne obrade) dovoljno brzo za primjenu na cijeloj slici.

In [ ]:
stereo_bm = cv2.StereoBM_create(numDisparities=32, blockSize=9)
disparity_bm = stereo_bm.compute(left, right).astype(np.float32) / 16.0  # fixed-point output, divide to get pixels

print('recovered disparity vs. ground truth, by region:')
print(f'  background (true=5):  {disparity_bm[100:110, 150:160].mean():.2f}')
print(f'  mid layer (true=15):  {disparity_bm[40:50, 60:70].mean():.2f}')
print(f'  near layer (true=25): {disparity_bm[70:80, 90:110].mean():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
im0 = axes[0].imshow(true_disparity, cmap='viridis', vmin=0, vmax=30)
axes[0].set_title('Ground truth')
im1 = axes[1].imshow(np.where(disparity_bm >= 0, disparity_bm, np.nan), cmap=invalid_cmap, vmin=0, vmax=30)
axes[1].set_title('cv2.StereoBM output\n(gray = invalid)')
for ax in axes:
    ax.axis('off')
plt.colorbar(im1, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

`StereoBM` označava piksel nevaljanim (vraća $-1$) kada nije siguran u jedinstveno podudaranje, primjerice preblizu rubu slike za pretraživanje cijelog raspona dispariteta. Ti su pikseli gore prikazani sivom bojom.

### Provjera dosljednosti ugrađena u `disp12MaxDiff`

Provjera dosljednosti lijevo–desno ugrađena je u `cv2.StereoBM` &mdash; postavite `disp12MaxDiff` na mali pozitivan broj (obično 1) kako biste je omogućili; zadana vrijednost $-1$ ostavlja je isključenom.

In [ ]:
stereo_checked = cv2.StereoBM_create(numDisparities=32, blockSize=9)
stereo_checked.setDisp12MaxDiff(1)
disparity_checked = stereo_checked.compute(left, right).astype(np.float32) / 16.0

newly_invalid = (disparity_bm >= 0) & (disparity_checked < 0)
print(f'valid pixels without the check: {(disparity_bm >= 0).sum()}')
print(f'valid pixels with the check:    {(disparity_checked >= 0).sum()}')
print(f'newly marked invalid:           {newly_invalid.sum()}')

plt.imshow(np.where(disparity_checked >= 0, disparity_checked, np.nan), cmap=invalid_cmap, vmin=0, vmax=30)
plt.colorbar(label='disparity (px)')
plt.title('cv2.StereoBM with disp12MaxDiff=1\n(gray = invalid)')
plt.axis('off')
plt.show()

## Veličina prozora: kompromis između detalja i šuma

Veći prozor za podudaranje usrednjava više piksela, čineći podudaranje otpornijim na šum, ali zamućuje preko dubinskih diskontinuiteta (miješa piksele dviju različitih stvarnih dubina u jednu „usrednjenu” procjenu dispariteta uz rubove objekata). Manji prozor čuva oštre dubinske granice, ali ga šum ili ponavljajuća tekstura lakše zavaraju.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, block_size in zip(axes, [5, 15, 31]):
    stereo = cv2.StereoBM_create(numDisparities=32, blockSize=block_size)
    d = stereo.compute(left, right).astype(np.float32) / 16.0
    im = ax.imshow(np.where(d >= 0, d, np.nan), cmap=invalid_cmap, vmin=0, vmax=30)
    ax.set_title(f'blockSize={block_size}', fontsize=9)
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

## Stvarni stereo par: Tsukuba

Dosad smo koristili sintetičke slike upravo kako bismo znali stvarne vrijednosti. Za stvarne fotografije ista tri prolaza (slijeva nadesno, zdesna nalijevo i provjera dosljednosti između njih) primjenjuju se kao i prije, ovaj put na stvarnom rektificiranom stereo paru fotografija.

Naizmjenični prikaz dvaju kadrova olakšava izravno uočavanje pomaka &mdash; promatrajte koliko se više pomiče svjetiljka nego police s knjigama iza nje, upravo pomak ovisan o dubini koji procjenjujemo kroz cijelu lekciju.

<img src="../img/tsukuba_flicker.gif" alt="Naizmjenični prikaz lijeve i desne slike stereo para Tsukuba" style="display: block; margin: 0 auto; max-width: 400px;">

<p class="image-attribution">Izvor slike: <a href="https://vision.middlebury.edu/stereo/data/scenes2001/" target="_blank" rel="noopener">Middlebury Stereo Datasets (Sveučilište u Tsukubi)</a></p>

In [ ]:
tsukuba_left = cv2.imread('../img/tsukuba_left.png', cv2.IMREAD_GRAYSCALE)
tsukuba_right = cv2.imread('../img/tsukuba_right.png', cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(tsukuba_left, cmap='gray')
axes[0].set_title('Left photo', fontsize=9)
axes[1].imshow(tsukuba_right, cmap='gray')
axes[1].set_title('Right photo', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
num_disp = 64  # must be divisible by 16; large enough that the nearest surfaces don't clip the search range
stereo = cv2.StereoBM_create(numDisparities=num_disp, blockSize=15)
disp_lr = stereo.compute(tsukuba_left, tsukuba_right).astype(np.float32) / 16.0
disp_rl = stereo.compute(tsukuba_right[:, ::-1], tsukuba_left[:, ::-1]).astype(np.float32) / 16.0
disp_rl = disp_rl[:, ::-1]

stereo_checked = cv2.StereoBM_create(numDisparities=num_disp, blockSize=15)
stereo_checked.setDisp12MaxDiff(1)
disp_checked = stereo_checked.compute(tsukuba_left, tsukuba_right).astype(np.float32) / 16.0

changed = disp_lr != disp_checked
print(f'pixels changed by the consistency check: {changed.sum()} / {changed.size} ({100 * changed.sum() / changed.size:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, d, title in zip(axes, [disp_lr, disp_rl, disp_checked],
                         ['Left-to-right disparity', 'Right-to-left disparity', 'After consistency check']):
    im = ax.imshow(np.where(d >= 0, d, np.nan), cmap=invalid_cmap, vmin=0, vmax=num_disp)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

Svjetiljka, najbliži objekt u prizoru, ističe se najvećim disparitetom; police s knjigama postupno se udaljavaju iza nje. Označeno je samo oko 1 % piksela, pa treći prikaz na prvi pogled izgleda gotovo isto kao prvi &mdash; no taj je točkasti šum koncentriran upravo uz dubinske diskontinuitete u cijelom prizoru: hrptove knjiga, rubove polica, siluetu glave, svugdje gdje prozor obuhvaća dvije različite dubine ili jedna kamera vidi nešto što druga ne vidi.

## Od dispariteta do karte dubine

Uz poznatu žarišnu duljinu i osnovicu (u bilo kojim dosljednim jedinicama), $Z = fB/d$ izravno pretvara kartu dispariteta u metričku kartu dubine.

In [ ]:
focal_length_px = 500.0
baseline_m = 0.1

valid = disparity_bm > 0
depth_m = np.full_like(disparity_bm, np.nan)
depth_m[valid] = focal_length_px * baseline_m / disparity_bm[valid]

plt.imshow(depth_m, cmap='viridis_r')
plt.colorbar(label='estimated depth (m)')
plt.title('Depth map (nearer = brighter)')
plt.axis('off')
plt.show()

In [ ]:
step = 3  # subsample for a readable, fast-to-render point cloud
ys, xs = np.mgrid[0:depth_m.shape[0]:step, 0:depth_m.shape[1]:step]
zs = depth_m[::step, ::step]
valid_pts = ~np.isnan(zs)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection='3d')
sc = ax.scatter(xs[valid_pts], ys[valid_pts], zs[valid_pts], c=zs[valid_pts], cmap='viridis_r', s=3)
ax.set_xlabel('x (px)')
ax.set_ylabel('y (px)')
ax.set_zlabel('depth (m)')
ax.invert_yaxis()  # match image row convention (row 0 at top)
ax.invert_zaxis()  # nearer (smaller depth) points appear higher, like the disparity peaks above
ax.view_init(elev=20, azim=-60)
#fig.colorbar(sc, ax=ax, shrink=0.6, label='depth (m)')
ax.set_title('The depth map, lifted into 3D')
plt.tight_layout()
plt.show()

Isti brojevi, prikazani kao točke koje lebde u prostoru umjesto kao boje na ravnoj slici, čine „ravnine” ovog sintetičkog prizora doslovnima: tri ravne terase na trima različitim visinama, s jasno izrezanim rupama ondje gdje `StereoBM` nije imao pouzdano podudaranje.

### Zadaci

1. Zamijenite nasumičnu teksturu slike `left` ravnomjernim sivim područjem u pozadini (zadržavajući teksturu dvaju pravokutnika). Ponovno pokrenite `cv2.StereoBM` i opišite što se događa s procjenom dispariteta na ravnomjernom području &mdash; to je problem otvora iz lekcije 21, sada u kontekstu stereo podudaranja.
2. Isprobajte `cv2.StereoSGBM_create` (poluglobalno podudaranje blokova, koje nameće glatkoću između susjednih dispariteta umjesto potpuno neovisnog podudaranja svakog piksela) umjesto `StereoBM`. Usporedite količinu točkastog šuma u ravnomjernim područjima i pozadini između tih dviju metoda.
3. Povećajte `baseline_m` u pretvorbi dubine. Kako se mijenja procijenjena karta dubine i zašto bi stereo postav s većim razmakom kamera davao preciznije procjene dubine udaljenih objekata (uz veće područje blizu kamera koje se ne može pouzdano izmjeriti, zbog raspona dispariteta i okluzije)?